# 销售数据分析示例

本笔记本演示如何读取和分析 Excel 中的销售数据。

In [31]:
# 导入必要的库
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

## 1. 读取 Excel 数据进行分析

演示如何从 Excel 文件中读取数据并进行基本分析。

In [32]:
# 读取 Excel 文件
excel_file = 'datas/sales_data.xlsx'

xls = pd.ExcelFile(excel_file)
print("可用工作表:", xls.sheet_names)

可用工作表: ['订单明细', '产品信息', '客户信息', '销售目标']


In [33]:
# 读取各工作表数据
orders = pd.read_excel(excel_file, sheet_name='订单明细')
products = pd.read_excel(excel_file, sheet_name='产品信息')
customers = pd.read_excel(excel_file, sheet_name='客户信息')
targets = pd.read_excel(excel_file, sheet_name='销售目标')

print("订单数据形状:", orders.shape)
print("产品数据形状:", products.shape)
print("客户数据形状:", customers.shape)

订单数据形状: (200, 11)
产品数据形状: (10, 5)
客户数据形状: (15, 4)


## 2. 基本数据分析

In [34]:
# 按区域统计销售额
region_sales = orders.groupby('region')['total_amount'].sum().sort_values(ascending=False)
print("\n=== 各区域销售额 ===")
print(region_sales)


=== 各区域销售额 ===
region
华东    104916.15
华南     66785.05
华中     32917.05
西北     31008.00
华北     29189.15
西南     26902.25
Name: total_amount, dtype: float64


In [35]:
# 按产品统计销量
product_sales = orders.groupby('product_id').agg({
    'quantity': 'sum',
    'total_amount': 'sum'
}).merge(products[['product_id', 'product_name']], on='product_id')
product_sales = product_sales.sort_values('total_amount', ascending=False)
print("\n=== 各产品销量 ===")
print(product_sales.to_string(index=False))


=== 各产品销量 ===
product_id  quantity  total_amount product_name
      P001        53     241201.75        笔记本电脑
      P003        51      14396.85         机械键盘
      P005        49       7051.65        显示器支架
      P007        58       6998.25         移动电源
      P009        43       6717.75        网络摄像头
      P006        25       5739.45         蓝牙耳机
      P010        62       3445.60        桌面收纳盒
      P004        49       3160.20      USB 集线器
      P002        37       1692.95         无线鼠标
      P008        28       1313.20         手机支架


In [36]:
# 按客户类型统计
customer_type_sales = orders.merge(customers[['customer_id', 'customer_type']], on='customer_id')
type_summary = customer_type_sales.groupby('customer_type')['total_amount'].agg(['sum', 'mean', 'count'])
type_summary.columns = ['总销售额', '平均订单金额', '订单数']
print("\n=== 按客户类型统计 ===")
print(type_summary)


=== 按客户类型统计 ===
                    总销售额       平均订单金额  订单数
customer_type                             
个人客户           140728.40  1421.498990   99
企业客户           150989.25  1494.943069  101


In [37]:
# 月度销售趋势
orders['month'] = orders['order_date'].dt.strftime('%Y-%m')
monthly_sales = orders.groupby('month')['total_amount'].sum()
print("\n=== 月度销售趋势 ===")
print(monthly_sales)


=== 月度销售趋势 ===
month
2025-01    46088.45
2025-02    24199.15
2025-03    52507.00
2025-04    53334.00
2025-05    71505.00
2025-06    44084.05
Name: total_amount, dtype: float64
